In [ ]:
import pandas as pd
import nltk
from nltk.corpus import sentiwordnet as swn
from nltk.corpus import wordnet as wn
from nltk.tokenize import word_tokenize

# Download necessary resources
nltk.download('punkt')
nltk.download('punkt_tab')  # Add this line
nltk.download('sentiwordnet')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

# 1. Bing Liu Sentiment Classifier (Built from scratch)
class BingLiuClassifier:
    def __init__(self, pos_file, neg_file):
        """Initializes the model by loading positive and negative dictionaries."""
        self.pos_words = self._load_lexicon(pos_file)
        self.neg_words = self._load_lexicon(neg_file)
        self.negation_words = {"not", "no", "never", "n't", "hardly", "barely", "neither", "nor", "none"}

    def _load_lexicon(self, file_path):
        """Reads the dictionary file and skips metadata/comments starting with ';'."""
        words = set()
        try:
            with open(file_path, 'r', encoding='iso-8859-1') as f:
                for line in f:
                    if not line.startswith(';') and line.strip():
                        words.add(line.strip().lower())
        except FileNotFoundError:
            print(f"Error: File '{file_path}' not found. Please check the path.")
        return words

    def analyze(self, text):
        """Analyzes text sentiment and handles negation logic."""
        if pd.isna(text): return "Neutral"
        
        tokens = word_tokenize(str(text).lower())
        score = 0
        
        for i, word in enumerate(tokens):
            word_sentiment = 0
            
            # Check if word exists in lexicons
            if word in self.pos_words:
                word_sentiment = 1
            elif word in self.neg_words:
                word_sentiment = -1
            
            # Negation Logic
            if i > 0 and tokens[i-1] in self.negation_words:
                word_sentiment *= -1
            
            score += word_sentiment
            
        return "Positive" if score > 0 else "Negative" if score < 0 else "Neutral"

# 2. SentiWordNet Classifier
class SentiWordNetClassifier:
    def get_wordnet_pos(self, tag):
        """Maps NLTK POS tags to WordNet POS tags."""
        if tag.startswith('J'): return wn.ADJ
        elif tag.startswith('V'): return wn.VERB
        elif tag.startswith('N'): return wn.NOUN
        elif tag.startswith('R'): return wn.ADV
        return None

    def analyze(self, text):
        """Calculates sentiment using SentiWordNet scores (PosScore - NegScore)."""
        if pd.isna(text): return "Neutral"
        
        tokens = word_tokenize(str(text))
        tagged_words = nltk.pos_tag(tokens)
        total_sentiment = 0
        
        for word, tag in tagged_words:
            wn_tag = self.get_wordnet_pos(tag)
            if wn_tag:
                synsets = list(swn.senti_synsets(word, wn_tag))
                if synsets:
                    senti_syn = synsets[0]
                    total_sentiment += (senti_syn.pos_score() - senti_syn.neg_score())
        
        # Use a small threshold to avoid bias
        if total_sentiment > 0.01: return "Positive"
        elif total_sentiment < -0.01: return "Negative"
        else: return "Neutral"

# 3. Execution on CSV Data
csv_file = 'full_clean_yt_comments.csv'  
text_column = 'clean_comment'
positive_lex = 'positive-words.txt'
negative_lex = 'negative-words.txt'

try:
    # Load the dataset
    print("Loading dataset...")
    df = pd.read_csv(csv_file)
    
    # Ensure the text column exists and handle missing values
    if text_column in df.columns:
        df[text_column] = df[text_column].fillna("").astype(str)
        
        # Initialize Models
        bing_model = BingLiuClassifier(positive_lex, negative_lex)
        swn_model = SentiWordNetClassifier()
        
        # Apply Analysis
        print("Processing sentiment analysis (this may take a moment)...")
        df['Bing_Liu_Result'] = df[text_column].apply(bing_model.analyze)
        df['SentiWordNet_Result'] = df[text_column].apply(swn_model.analyze)
        
        # Save results to a new CSV file
        output_name = 'sentiment_analysis_output.csv'
        df.to_csv(output_name, index=False)
        print(f"Success! Resul saved to '{output_name}'")
        
        # Preview the results
        print("\n--- Preview of Results ---")
        print(df[[text_column, 'Bing_Liu_Result', 'SentiWordNet_Result']].head())
    else:
        print(f"Error: Column '{text_column}' not found in {csv_file}.")

except Exception as e:
    print(f"An error occurred: {e}")